# 云计算技术 · 第一次课后作业（**学生版**），截止日期为4月19日，发送助教王宁邮箱：1654053734@qq.com
## StatefulSet 部署 Redis + Python 状态读取

---
**姓名**：___________　　**学号**：___________　　**完成日期**：___________

---
| 任务 | 内容 | 分值 |
|------|------|------|
| **任务一** | 编写 StatefulSet YAML（Headless Service + StatefulSet + volumeClaimTemplates） | **50 分** |
| **任务二** | Python 读取 StatefulSet 状态（4 个函数） | **50 分** |
| 加分项 A | ConfigMap 管理 Redis 配置 | +5 分 |
| 加分项 B | tabulate 表格化输出 | +3 分 |
| 加分项 C | 完善异常处理 | +2 分 |

> ⚡ **作答规则**：将所有 `???` 替换为正确内容；不要修改函数名和参数；按顺序从上到下执行单元格。

---
## 📦 模块 0：依赖说明（请先阅读）

### 系统依赖（需手动安装）

| 工具 | 作用 | 安装文档 |
|------|------|---------|
| **kubectl** | Kubernetes 命令行工具，用于与集群交互 | [官方文档](https://kubernetes.io/docs/tasks/tools/) |
| **Minikube**（推荐） | 本地单节点 K8s 集群 | [官方文档](https://minikube.sigs.k8s.io/docs/start/) |
| **Docker**（Minikube 依赖） | Minikube 的默认驱动 | [官方文档](https://docs.docker.com/engine/install/) |

> **快速安装命令（macOS）：** `brew install minikube kubectl`
>
> **快速安装命令（Linux）：** 运行下方【模块 1】的自动安装单元格

### Python 依赖（自动安装）

| 包名 | 版本要求 | 用途 |
|------|---------|------|
| `kubernetes` | ≥ 26.1 | 官方 Kubernetes Python 客户端 |
| `tabulate` | ≥ 0.9 | 表格化输出（加分项 B） |
| `nbformat` | ≥ 5.0 | Notebook 解析（**仅教师版**批量评分使用） |
| `nbconvert` | ≥ 7.0 | Notebook 执行（**仅教师版**批量评分使用） |

### 运行顺序

```
模块 0（阅读） → 模块 1（一键配置） → 任务一 → 任务二 → 加分项 → 评分检查
```

> ⚠️ **务必按顺序执行单元格**，后续任务依赖前面步骤的变量和函数定义。

---
## 模块 1：一键环境配置

运行此单元格，自动安装 Python 依赖、检测并启动集群。

In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  模块 1：一键环境配置                                         ║
# ║  运行此单元格，自动完成 Python 依赖安装、集群检测与启动          ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys, platform, shutil, time, os

OS = platform.system()   # 'Darwin' | 'Linux' | 'Windows'

# ── 工具函数 ──────────────────────────────────────────────────────
def run(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(r.stderr.strip())
    return r.stdout.strip(), r.stderr.strip(), r.returncode

def pip_install(*pkgs):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *pkgs, "-q",
         "--disable-pip-version-check"],
        stdout=subprocess.DEVNULL
    )

def ok(msg):  print(f"  ✅ {msg}")
def warn(msg): print(f"  ⚠️  {msg}")
def err(msg):  print(f"  ❌ {msg}")
def section(title): print(f"\n{'─'*50}\n  {title}\n{'─'*50}")

# ══ 步骤 1：安装 Python 包 ════════════════════════════════════════
section("步骤 1/4  安装 Python 依赖")
for pkg, import_name in [("kubernetes","kubernetes"), ("tabulate","tabulate"),
                          ("nbformat","nbformat"),   ("nbconvert","nbconvert")]:
    try:
        __import__(import_name)
        ok(f"{pkg} 已安装")
    except ImportError:
        print(f"  📦 正在安装 {pkg}...", end=" ", flush=True)
        pip_install(pkg)
        ok(f"{pkg} 安装完成")

# ══ 步骤 2：检测 kubectl ══════════════════════════════════════════
section("步骤 2/4  检测 kubectl")
if shutil.which("kubectl"):
    out, _, _ = run("kubectl version --client --short 2>/dev/null || kubectl version --client")
    ok(f"kubectl 已就绪  ({out.split(chr(10))[0][:60]})")
else:
    err("未检测到 kubectl")
    if OS == "Darwin":
        print("  👉 macOS 安装命令：brew install kubectl")
    elif OS == "Linux":
        print("  👉 Linux 自动安装中...")
        cmds = [
            'curl -LO "https://dl.k8s.io/release/$(curl -Ls https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"',
            "chmod +x kubectl",
            "mkdir -p ~/.local/bin && mv kubectl ~/.local/bin/kubectl",
        ]
        success = True
        for c in cmds:
            _, e, rc = run(c)
            if rc != 0:
                warn(f"命令失败，请手动安装 kubectl：{e[:80]}")
                success = False; break
        if success:
            os.environ["PATH"] = os.environ["PATH"] + ":~/.local/bin"
            ok("kubectl 安装完成，已加入 PATH")
    else:
        print("  👉 Windows 安装：https://kubernetes.io/docs/tasks/tools/install-kubectl-windows/")

# ══ 步骤 3：检测并安装 Minikube ════════════════════════════════════
section("步骤 3/4  检测 Minikube")
if shutil.which("minikube"):
    out, _, _ = run("minikube version")
    ok(f"Minikube 已就绪  ({out.split(chr(10))[0][:60]})")
else:
    warn("未检测到 Minikube")
    if OS == "Darwin":
        print("  👉 macOS 安装命令：brew install minikube")
        print("  👉 或访问：https://minikube.sigs.k8s.io/docs/start/")
    elif OS == "Linux":
        print("  📦 Linux 正在自动下载 Minikube...")
        cmds = [
            "curl -LO https://storage.googleapis.com/minikube/releases/latest/minikube-linux-amd64",
            "chmod +x minikube-linux-amd64",
            "mkdir -p ~/.local/bin && mv minikube-linux-amd64 ~/.local/bin/minikube",
        ]
        success = True
        for c in cmds:
            _, e, rc = run(c)
            if rc != 0:
                warn(f"自动安装失败：{e[:80]}")
                print("  👉 请手动安装：https://minikube.sigs.k8s.io/docs/start/")
                success = False; break
        if success:
            ok("Minikube 下载完成")
    else:
        print("  👉 Windows：https://minikube.sigs.k8s.io/docs/start/")

# ══ 步骤 4：启动 / 检测集群状态 ════════════════════════════════════
section("步骤 4/4  检测 Kubernetes 集群")
out, _, rc = run("kubectl cluster-info 2>/dev/null")
if rc == 0:
    ok(f"集群已就绪：{out.split(chr(10))[0][:70]}")
else:
    warn("集群未运行，尝试启动 Minikube...")
    if shutil.which("minikube"):
        print("  ⏳ minikube start（首次启动需 1-3 分钟）...", flush=True)
        out2, err2, rc2 = run("minikube start --driver=docker 2>&1")
        if rc2 == 0:
            ok("Minikube 启动成功！")
            out3, _, _ = run("kubectl cluster-info")
            print(f"  {out3.split(chr(10))[0][:70]}")
        else:
            err("Minikube 启动失败，错误如下：")
            print(f"  {err2[:300]}")
            print("  👉 尝试：minikube start --driver=virtualbox  或检查 Docker 是否运行")
    else:
        err("Minikube 未安装，请先完成步骤 3 后重新运行此单元格")

# ══ 全局初始化 run() 供后续单元格使用 ═════════════════════════════
print("\n" + "═"*50)
print("  环境配置完成，run() 函数已就绪，可继续执行后续单元格")
print("═"*50)


──────────────────────────────────────────────────
  步骤 1/4  安装 Python 依赖
──────────────────────────────────────────────────
  ✅ kubernetes 已安装
  ✅ tabulate 已安装
  ✅ nbformat 已安装
  ✅ nbconvert 已安装

──────────────────────────────────────────────────
  步骤 2/4  检测 kubectl
──────────────────────────────────────────────────
  ✅ kubectl 已就绪  (Client Version: v1.35.3)

──────────────────────────────────────────────────
  步骤 3/4  检测 Minikube
──────────────────────────────────────────────────
  ✅ Minikube 已就绪  (minikube version: v1.38.1)

──────────────────────────────────────────────────
  步骤 4/4  检测 Kubernetes 集群
──────────────────────────────────────────────────
  ⚠️  集群未运行，尝试启动 Minikube...
  ⏳ minikube start（首次启动需 1-3 分钟）...
  ❌ Minikube 启动失败，错误如下：
  
  👉 尝试：minikube start --driver=virtualbox  或检查 Docker 是否运行

══════════════════════════════════════════════════
  环境配置完成，run() 函数已就绪，可继续执行后续单元格
══════════════════════════════════════════════════


---
## 任务一：编写 StatefulSet YAML（50 分）

### 评分细则速查

| 子项 | 分值 | 关键点 |
|------|------|-------|
| Headless Service（`clusterIP: None`） | 10 | 没有这个，StatefulSet Pod 的 DNS 无法解析 |
| StatefulSet 基础结构 + `replicas: 3` | 10 | selector 与 template labels 必须一致 |
| `serviceName` 字段 | 5 | 必须与 Service name 完全一致 |
| `resources`（requests + limits） | 10 | 单位要对（m、Mi、Gi） |
| `volumeClaimTemplates` + `/data` 挂载 | 15 | 注意不是普通 `volumes`，每个 Pod 独立 PVC |

### 🖊️ 填写你的 YAML

In [ ]:
YAML_CONTENT = '''
# ──────────────────────────────────────────────────────────────
#  将所有 ??? 替换为正确内容，保持缩进格式不变
# ──────────────────────────────────────────────────────────────

# 资源一：Headless Service（StatefulSet 必须配套）
apiVersion: v1
kind: Service
metadata:
  name: ???              # Service 名称（与 StatefulSet 的 serviceName 保持一致）
  namespace: default
spec:
  clusterIP: ???         # Headless 模式的特殊值是什么？
  selector:
    app: ???
  ports:
    - port: ???          # Redis 默认端口号
      targetPort: ???
---
# 资源二：StatefulSet
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: ???
  namespace: default
spec:
  serviceName: ???       # 填写上方 Service 的 name
  replicas: ???          # 部署副本数
  selector:
    matchLabels:
      app: ???
  template:
    metadata:
      labels:
        app: ???
    spec:
      containers:
        - name: redis
          image: ???     # 填写镜像（推荐 redis:7.0）
          ports:
            - containerPort: ???
          resources:
            requests:
              cpu: ???   # 最小 CPU（例：100m）
              memory: ??? # 最小内存（例：128Mi）
            limits:
              cpu: ???
              memory: ???
          volumeMounts:
            - name: redis-data
              mountPath: ???  # Redis 持久化数据目录（提示：/data）
  volumeClaimTemplates:
    - metadata:
        name: redis-data
      spec:
        accessModes:
          - ???          # 单节点读写，三个单词
        resources:
          requests:
            storage: ???  # 申请 1Gi
'''
print("YAML 已定义，请确认所有 ??? 已替换，再运行下方单元格")

### 验证：dry-run + 部署 + 等待就绪

In [ ]:
# 语法预检（dry-run）+ 部署
YAML_FILE = "/tmp/redis-statefulset.yaml"
with open(YAML_FILE, "w") as f:
    f.write(YAML_CONTENT)
out, err, c = run(f"kubectl apply -f {YAML_FILE} --dry-run=client")
if c != 0:
    print("❌ YAML 语法错误："); print(err)
else:
    print("✅ dry-run 通过，开始部署...")
    out2, err2, c2 = run(f"kubectl apply -f {YAML_FILE}")
    print(out2 if c2 == 0 else err2)

In [ ]:
import time
print("⏳ 等待 Pod 就绪（有序启动，约 30-90 秒）...")
for i in range(30):
    out, _, _ = run("kubectl get statefulset redis -o jsonpath='{.status.readyReplicas}' 2>/dev/null")
    ready = int(out.strip("'") or 0)
    print(f"  [{i*5:3d}s] 就绪: {ready}/3", end="\r")
    if ready == 3:
        print(f"\n✅ 全部 3 个 Pod 已就绪！"); break
    time.sleep(5)
else:
    print("\n⚠️ 超时，请检查：kubectl get pods -l app=redis")
print()
out, _, _ = run("kubectl get pods -l app=redis -o wide")
print(out)

In [ ]:
print("=== ① Pod 命名稳定性 ===")
out, _, _ = run("kubectl get pods -l app=redis --no-headers -o custom-columns=NAME:.metadata.name 2>/dev/null")
pods = [p for p in out.strip().split('\n') if p]
if set(pods) == {"redis-0","redis-1","redis-2"}:
    print(f"✅ {sorted(pods)}")
else:
    print(f"❌ 实际: {pods}")
print("\n=== ② PVC（每 Pod 独立）===")
out, _, _ = run("kubectl get pvc 2>/dev/null")
print(out or "（无 PVC）")
print("\n=== ③ Headless Service ===")
out, _, _ = run("kubectl get svc redis-headless 2>/dev/null")
print(out or "（未找到 redis-headless）")

---
## 任务二：Python 读取 StatefulSet 状态（50 分）

### 评分细则速查

| 步骤 | 分值 | 核心 API |
|------|------|---------|
| 第1步：客户端初始化 | 10 | `config.load_kube_config()` + 两个 `client.*Api()` |
| 第2步：读取 StatefulSet | 10 | `apps_v1.read_namespaced_stateful_set(name, namespace)` |
| 第3步：打印副本状态 | 15 | `spec.replicas` / `status.ready_replicas` / `status.current_replicas` |
| 第4步：遍历 Pod | 15 | `core_v1.list_namespaced_pod(namespace, label_selector=...)` |

### 🖊️ 第1步：初始化客户端（10 分）

In [ ]:
from kubernetes import client, config
from kubernetes.client.exceptions import ApiException

# ─────────────────────────────────────────────────────────────────
# 【任务二 · 第1步】初始化 Kubernetes 客户端          （10分）
#
# 需要用到：
#   config.load_kube_config()   加载 ~/.kube/config
#   client.AppsV1Api()          管理 apps/v1 资源（StatefulSet 所在 API 组）
#   client.CoreV1Api()          管理核心资源（Pod、Service）
# ─────────────────────────────────────────────────────────────────
def init_k8s_client():
    """加载 kubeconfig，返回 (apps_v1, core_v1) 两个 API 客户端对象"""
    try:
        ???  # 加载本地 kubeconfig
    except config.ConfigException:
        config.load_incluster_config()   # Pod 内运行时使用此分支，无需修改

    apps_v1 = ???  # 创建 AppsV1Api 对象
    core_v1 = ???  # 创建 CoreV1Api 对象
    return apps_v1, core_v1

apps_v1, core_v1 = init_k8s_client()
print("✅ 初始化成功" if (apps_v1 and core_v1) else "❌ 初始化失败，检查函数实现")

### 🖊️ 第2步：读取 StatefulSet（10 分）

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 【任务二 · 第2步】读取 StatefulSet 对象              （10分）
#
# 关键 API：
#   apps_v1.read_namespaced_stateful_set(name=..., namespace=...)
#   成功返回 StatefulSet 对象；失败时捕获 ApiException 并返回 None
# ─────────────────────────────────────────────────────────────────
def get_statefulset_info(apps_v1, name="redis", namespace="default"):
    try:
        sts = ???   # 调用 API 读取 StatefulSet
        return sts
    except ApiException as e:
        print(f"[错误] {e.status} {e.reason}")
        return None

sts = get_statefulset_info(apps_v1)
print("✅ 读取成功：", sts.metadata.name) if sts else print("❌ 读取失败")

### 🖊️ 第3步：打印副本状态（15 分）

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 【任务二 · 第3步】打印副本状态                        （15分）
#
# 需要打印的字段（所有 or 0 处理防止 None 报错）：
#   sts.metadata.name                  名称
#   sts.metadata.namespace             命名空间
#   sts.metadata.creation_timestamp    创建时间
#   sts.spec.replicas                  期望副本数
#   sts.status.ready_replicas          就绪副本数（可能 None）
#   sts.status.current_replicas        当前副本数（可能 None）
# ─────────────────────────────────────────────────────────────────
def print_replica_status(sts):
    if sts is None: return
    print("=" * 50)
    print("  StatefulSet 基本信息")
    print("=" * 50)
    print(f"  名称         : ???")   # ← 填字段
    print(f"  命名空间     : ???")   # ← 填字段
    print(f"  创建时间     : ???")   # ← 填字段
    print()
    print("  副本状态")
    print("-" * 50)
    print(f"  期望副本数   : ???")   # ← spec.replicas
    print(f"  就绪副本数   : ???")   # ← status.ready_replicas or 0
    print(f"  当前副本数   : ???")   # ← status.current_replicas or 0

print_replica_status(sts)

### 🖊️ 第4步：遍历 Pod 状态（15 分）

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 【任务二 · 第4步】遍历打印 Pod 状态                  （15分）
#
# 关键 API：
#   core_v1.list_namespaced_pod(namespace=..., label_selector=...)
#   返回对象的 .items 是 Pod 列表，按 metadata.name 排序后遍历
#
# 每个 Pod 需打印：
#   pod.metadata.name    Pod 名称
#   pod.status.phase     运行阶段（可能 None → 用 or 'Unknown'）
#   pod.status.pod_ip    IP 地址（可能 None → 用 or 'N/A'）
# ─────────────────────────────────────────────────────────────────
def print_pod_status(core_v1, namespace="default", label_selector="app=redis"):
    try:
        pods = ???   # 调用 API 列出 Pod（传入 namespace 和 label_selector）
    except ApiException as e:
        print(f"[错误] {e.reason}"); return

    print()
    print("  Pod 状态明细")
    print("-" * 50)
    print(f"  {'Pod 名称':<30} {'Phase':<12} Pod IP")
    print(f"  {'-'*28:<30} {'-'*10:<12} ------")
    for pod in sorted(pods.items, key=lambda p: p.metadata.name):
        print(f"  ???")   # ← 填写打印内容（参考列头格式）
    print("=" * 50)

print_pod_status(core_v1)

### 整合：一键输出

In [ ]:
def read_all_status():
    a, c = init_k8s_client()
    sts = get_statefulset_info(a)
    print_replica_status(sts)
    print_pod_status(c)
read_all_status()

---
## 加分项（可选，最多 +10 分）

### 加分项 A：ConfigMap 管理 Redis 配置（+5 分）

**提示**：`ConfigMap.data` 存配置内容；StatefulSet 用 `volumes` + `volumeMounts` 挂载；启动命令改为 `redis-server /etc/redis/redis.conf`

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 【加分项 A】ConfigMap 管理 Redis 配置（+5分）
#
# 要求：
#   1. 定义含 redis.conf 的 ConfigMap（data 字段）
#   2. StatefulSet 中用 volumes + volumeMounts 挂载
#   3. 容器启动命令改为 redis-server /etc/redis/redis.conf
# ─────────────────────────────────────────────────────────────────
CONFIGMAP_YAML = '''
???  # 在此填写 ConfigMap YAML
'''
with open("/tmp/redis-config.yaml", "w") as f:
    f.write(CONFIGMAP_YAML)
out, err, c = run("kubectl apply -f /tmp/redis-config.yaml")
print(out if c == 0 else err)

> **实现思路**（填写）：
> ...

### 加分项 B：tabulate 表格化（+3 分）

In [ ]:
from tabulate import tabulate
# ─────────────────────────────────────────────────────────────────
# 【加分项 B】tabulate 表格化输出（+3分）
# 要求：表格包含 Pod名称、Phase、IP、节点、启动时间
# 提示：pod.status.start_time 为 datetime（可能 None）
# ─────────────────────────────────────────────────────────────────
def print_pod_table(core_v1, namespace="default", label_selector="app=redis"):
    pods = core_v1.list_namespaced_pod(namespace=namespace, label_selector=label_selector)
    rows = []
    for pod in sorted(pods.items, key=lambda p: p.metadata.name):
        rows.append([
            pod.metadata.name,
            pod.status.phase or "Unknown",
            pod.status.pod_ip or "N/A",
            pod.spec.node_name or "N/A",
            ???   # 填写：启动时间（处理 None）
        ])
    print(tabulate(rows, headers=["Pod名称","Phase","IP","节点","启动时间"], tablefmt="grid"))
print_pod_table(core_v1)

### 加分项 C：完善异常处理（+2 分）

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 【加分项 C】完善异常处理（+2分）
# 要求：区分 404/403 及其他 HTTP 错误，额外捕获网络层异常
# ─────────────────────────────────────────────────────────────────
def get_statefulset_info_v2(apps_v1, name="redis", namespace="default"):
    try:
        return apps_v1.read_namespaced_stateful_set(name=name, namespace=namespace)
    except ApiException as e:
        if e.status == 404:
            print(f"[404] ???")   # 提示资源不存在
        elif e.status == 403:
            print(f"[403] ???")   # 提示权限不足
        else:
            print(f"[{e.status}] ???")
        return None
    except Exception as e:
        print(f"[网络异常] ???")  # 打印异常类型和信息
        return None

print(get_statefulset_info_v2(apps_v1, name="not-exist"))
print(get_statefulset_info_v2(apps_v1, name="redis"))

---
## 自动评分检查

运行后查看得分报告（不影响集群状态）。

In [ ]:
# AUTOSCORE_BEGIN
score = 0
results = []

def check(item, cond, pts, hint=""):
    global score
    ok = bool(cond)
    if ok: score += pts
    results.append(("✅" if ok else "❌", item, pts, ok, hint))

# ── 任务一：YAML 静态分析 ──────────────────────────────────────────
y = YAML_CONTENT

check("① Headless Service（clusterIP: None）",
      "kind: Service" in y and "clusterIP: None" in y, 10,
      "需要包含 clusterIP: None 的 Service 资源")
check("② StatefulSet（replicas: 3）",
      "kind: StatefulSet" in y and "replicas: 3" in y, 10,
      "需要 kind: StatefulSet 且 replicas: 3")
check("③ serviceName 非占位符",
      "serviceName:" in y and "???" not in y.split("serviceName:")[1][:40], 5,
      "serviceName 必须填写实际值")
check("④ resources（requests + limits）",
      "requests:" in y and "limits:" in y and "cpu:" in y and "memory:" in y, 10,
      "需配置 requests 和 limits")
check("⑤ volumeClaimTemplates + /data",
      "volumeClaimTemplates:" in y and "ReadWriteOnce" in y and "/data" in y, 15,
      "需要 volumeClaimTemplates、ReadWriteOnce、/data 挂载点")

# ── 任务二：运行时验证 ─────────────────────────────────────────────
try:
    _a, _c = init_k8s_client()
    check("⑥ K8s 客户端初始化",
          _a is not None and _c is not None, 10,
          "init_k8s_client() 需返回两个非 None 客户端")
    _sts = get_statefulset_info(_a) if _a else None
    check("⑦ 读取 StatefulSet 对象",
          _sts is not None and _sts.metadata.name == "redis", 10,
          "get_statefulset_info() 未返回有效对象")
    check("⑧ spec.replicas == 3",
          _sts is not None and _sts.spec.replicas == 3, 15,
          "spec.replicas 应为 3")
    if _c:
        _pods = _c.list_namespaced_pod(namespace="default", label_selector="app=redis")
        check("⑨ Pod 列表（3 个）",
              len(_pods.items) == 3, 15,
              f"读到 {len(_pods.items)} 个 Pod，期望 3")
    else:
        check("⑨ Pod 列表（3 个）", False, 15, "CoreV1Api 未初始化")
except Exception as e:
    for lbl, pts in [("⑥ K8s 客户端初始化",10),("⑦ 读取 StatefulSet 对象",10),
                     ("⑧ spec.replicas == 3",15),("⑨ Pod 列表（3 个）",15)]:
        check(lbl, False, pts, str(e)[:60] if lbl.startswith("⑥") else "")

# ── 输出报告 ───────────────────────────────────────────────────────
SEP = "═"*62
print(f"\n{SEP}")
print(f"  自动评分报告")
print(f"{SEP}")
print(f"  {'检查项':<36} {'分值':>4}   结果")
print(f"  {'─'*34:<36} {'─'*4}   ────")
for status, item, pts, ok, hint in results:
    note = f"  ← {hint}" if not ok and hint else ""
    print(f"  {status} {item:<34} {pts:>3}分   {'得分' if ok else '未得分'}{note}")
print(f"{SEP}")
print(f"  必做题合计：{score} / 100 分")
print(f"  加分项（A/B/C）需教师人工评分，最多 +10 分")
print(f"{SEP}")
# AUTOSCORE_END

---
## 清理资源
⚠️ 请先截图/保存输出，再运行清理。

In [ ]:
# ⚠️ 请先截图保存输出，再运行此单元格
run("kubectl delete statefulset redis --ignore-not-found=true")
run("kubectl delete svc redis-headless --ignore-not-found=true")
print("StatefulSet 和 Service 已删除")
print("\n剩余 PVC（需手动清理）：")
out, _, _ = run("kubectl get pvc -l app=redis 2>/dev/null")
print(out or "（无）")
# 如需一并删除 PVC：
# run("kubectl delete pvc -l app=redis")

---
**提交格式**：另存为 `学号_姓名_hw5.ipynb` 后上传课程平台。

*云计算技术课程组 · 第五讲*